Objetivos da Análise:

1: Identificar Gargalos Regionais: Mapear quais estados ou cidades possuem o maior volume de reclamações para entender falhas logísticas.

2: Análise de Eficiência de Atendimento: Avaliar a taxa de resolução (Status) das reclamações para medir a qualidade do pós-venda.

3: Sazonalidade de Problemas: Verificar se há picos de reclamações em datas específicas (como Black Friday ou Natal) usando a coluna de Tempo.

4: Categorização de Falhas: Descobrir quais os temas (TEMA) mais recorrentes para sugerir melhorias em processos internos.

##Primeira parte: Reconhecimento dos dados


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import re
import unicodedata
import plotly.express as px
import plotly.graph_objects as go

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#df = pd.read_csv("RECLAMEAQUI_NAGEM.csv")
#df.head()
caminho = '/content/drive/MyDrive/Tulio amigo/RECLAMEAQUI_NAGEM.csv'

# 3. Carregar o arquivo CSV
try:
    df = pd.read_csv(caminho)
    print("Arquivo carregado com sucesso!")

#     # Exibe as primeiras 5 linhas para conferir os dados
    display(df.head())
except FileNotFoundError:
    print("Erro: O arquivo não foi encontrado. Verifique se o nome da pasta ou do arquivo está correto.")

Arquivo carregado com sucesso!


,ID,TEMA,LOCAL,TEMPO,CATEGORIA,STATUS,DESCRICAO,URL,ANO,MES,DIA,DIA_DO_ANO,SEMANA_DO_ANO,DIA_DA_SEMANA,TRIMETRES,CASOS
0,27859289,Seguro Nagem celular é uma [Editado pelo Recla...,Fortaleza - CE,2017-01-08,Nagem - Loja Física<->Problemas com o Atendimento,Não resolvido,Comprei um celular LG na loja NAGEM do shoppin...,https://www.reclameaqui.com.br//nagem-loja-fis...,2017,1,8,8,1,6,1,1
1,28562465,Demora na finalização da venda / Mau atendimento,Recife - PE,2017-01-09,Nagem - Loja Física,Resolvido,"Fui na loja Nagem do Shopping Recife, fiz uma ...",https://www.reclameaqui.com.br//nagem-loja-fis...,2017,1,9,9,2,0,1,1
2,29249337,Problemas com a troca de produto,Salvador - BA,2017-01-10,Nagem - Loja Física,Respondida,Comprei um celular da Samsung J5 prime no sába...,https://www.reclameaqui.com.br//nagem-loja-fis...,2017,1,10,10,2,1,1,1
3,29999183,Produto com defeito/desrespeito com o consumidor,Olinda - PE,2017-01-11,Nagem - Loja Física,Respondida,No dia 25 /10 /2017 realizei uma compra de um ...,https://www.reclameaqui.com.br//nagem-loja-fis...,2017,1,11,11,2,2,1,1
4,30812099,Descaso Comsumidor,Salvador - BA,2017-01-12,TV<->Blackfriday<->Tela trincada<->Eletroeletr...,Não resolvido,"Na sexta feira do dia 24/11/2017,fui efetuar u...",https://www.reclameaqui.com.br//nagem-loja-fis...,2017,1,12,12,2,3,1,1


In [ ]:
def extract_city_state_from_cleaned_local(local_str):
    """
    Extrai o nome da cidade e a UF.
    Transforma nomes compostos apenas por hifens/espaços em 'Não Informado'.
    """
    if not isinstance(local_str, str) or 'desconhecido' in local_str.lower():
        return None, None

    # Limpeza inicial para remover espaços extras
    local_str = local_str.strip()
    parts = local_str.rsplit(' ', 1)

    if len(parts) == 2:
        city_raw = parts[0].strip()
        state = parts[1].strip().upper()

        # Validação da UF (deve ter 2 letras)
        if len(state) == 2 and state.isalpha():
            # Verifica se a cidade é vazia ou contém apenas hifens e espaços
            # O regex r'^[-\s]+$' captura strings como "-", "--", "- -", etc.
            if not city_raw or re.match(r'^[-\s]+$', city_raw):
                city = 'Não Informado'
            else:
                city = city_raw.title()

            return city, state

    # Caso a entrada seja apenas a UF (ex: 'MA' ou 'ma')
    elif len(parts) == 1:
        state = parts[0].strip().upper()
        if len(state) == 2 and state.isalpha():
            return 'Não Informado', state

    return None, None

# Criar o dicionário
city_to_state_from_df = {}
for local_entry in df['LOCAL'].unique():
    city, state = extract_city_state_from_cleaned_local(local_entry)
    if city and state:
        # Keep the accent marks here!
        # .title() preserves the proper formatting (e.g., "São Paulo")
        city_to_state_from_df[city] = state

def limpar_texto(txt):
    if not isinstance(txt, str): return str(txt)
    nfkd = unicodedata.normalize('NFKD', txt)
    return "".join([c for c in nfkd if not unicodedata.combining(c)]).lower().strip()

# 2. Criar a tupla 'estados_com_acento'
# Assumindo que queremos os nomes próprios para exibição (ex: Recife - PE)
# Vou capturar os dados atuais que já estão formatados ou os originais se preferir
estados_com_acento = tuple(city_to_state_from_df)

# 3. Criar a tupla 'estados_padronizados'
# Removendo acentos e padronizando para minúsculas para análise, incluindo a UF
estados_padronizados = tuple([limpar_texto(f"{city_part} {state_abbr}") for city_part, state_abbr in city_to_state_from_df.items()])

print(estados_com_acento)


('Fortaleza -', 'Recife -', 'Salvador -', 'Olinda -', 'São José De Mipibu -', 'Jaboatão Dos Guararapes -', 'Abreu E Lima -', 'Aracaju -', 'Beberibe -', 'Cupira -', 'Feira De Santana -', 'Aliança -', 'Brumado -', 'São Paulo -', 'Juazeiro -', 'Natal -', 'Itabaiana -', 'Paulista -', 'Petrolina -', 'João Pessoa -', 'Vitória De Santo Antão -', 'Caruaru -', 'Aquidabã -', 'Arcoverde -', 'Bom Jesus Da Lapa -', 'Sete Lagoas -', 'Rio De Janeiro -', 'São Cristóvão -', 'São José De Ribamar -', 'Bezerros -', 'Nísia Floresta -', 'São Bernardo Do Campo -', 'Curaçá -', 'Cabo De Santo Agostinho -', 'Petrolândia -', 'Eusébio -', 'Belo Jardim -', 'Campina Grande -', 'Ananindeua -', 'Ruy Barbosa -', 'São Luís -', 'Florianópolis -', 'São Caitano -', 'Teresina -', 'Sento Sé -', 'Belém -', 'Sobral -', 'Osasco -', 'Camaragibe -', 'Parnamirim -', 'Ipatinga -', 'Machados -', 'Barra Dos Coqueiros -', 'Glória Do Goitá -', 'Mossoró -', 'Patos -', 'Maracanaú -', 'Senhor Do Bonfim -', 'Almeirim -', 'Salgueiro -', 'S

In [ ]:
print(estados_padronizados)
# Se 'LOCAL' é a coluna original que você limpou:
df['LABEL'] = df['LOCAL'].apply(lambda x: extract_city_state_from_cleaned_local(x)[0])
df['LABEL'] = df['LABEL'].map(
    lambda x: f"{x} - {city_to_state_from_df[x]}" if x in city_to_state_from_df else x
)

('fortaleza - ce', 'recife - pe', 'salvador - ba', 'olinda - pe', 'sao jose de mipibu - rn', 'jaboatao dos guararapes - pe', 'abreu e lima - pe', 'aracaju - se', 'beberibe - ce', 'cupira - pe', 'feira de santana - ba', 'alianca - pe', 'brumado - ba', 'sao paulo - sp', 'juazeiro - ba', 'natal - rn', 'itabaiana - se', 'paulista - pe', 'petrolina - pe', 'joao pessoa - pb', 'vitoria de santo antao - pe', 'caruaru - pe', 'aquidaba - se', 'arcoverde - pe', 'bom jesus da lapa - ba', 'sete lagoas - mg', 'rio de janeiro - rj', 'sao cristovao - se', 'sao jose de ribamar - ma', 'bezerros - pe', 'nisia floresta - rn', 'sao bernardo do campo - sp', 'curaca - ba', 'cabo de santo agostinho - pe', 'petrolandia - pe', 'eusebio - ce', 'belo jardim - pe', 'campina grande - pb', 'ananindeua - pa', 'ruy barbosa - ba', 'sao luis - ma', 'florianopolis - sc', 'sao caitano - pe', 'teresina - pi', 'sento se - ba', 'belem - al', 'sobral - ce', 'osasco - sp', 'camaragibe - pe', 'parnamirim - rn', 'ipatinga - mg',

In [ ]:
# 2. Definir a função de padronização específica para a coluna LOCAL
def formatar_localidade(texto):
    if not isinstance(texto, str) or texto.strip() == '':
        return 'Desconhecido - Desconhecido'

    # Limpar espaços extras
    texto = texto.strip()

    # Tentar identificar se termina com sigla de estado (2 letras)
    # Exemplo: 'recife pe' ou 'fortaleza  ce'
    partes = texto.split()
    if len(partes) >= 2:
        uf = partes[-1].upper()
        # Verifica se a última parte tem 2 letras (padrão de UF)
        if len(uf) == 2 and uf.isalpha():
            cidade = " ".join(partes[:-1]).title()
            return f"{cidade} - {uf}"

    # Caso não siga o padrão 'cidade uf', apenas capitaliza
    return texto.title()

# 3. Aplicar apenas na coluna LOCAL
df['LOCAL'] = df['LOCAL'].apply(extract_city_state_from_cleaned_local)

# 4. Validar resultado
print("Coluna 'LOCAL' padronizada com sucesso!")
display(df[['LOCAL']].head(15))
print(f"\nTotal de localidades únicas: {df['LOCAL'].nunique()}")

print('\nFrequência das localidades (Top 20):')
print(df['LOCAL'].value_counts().head(20))

Coluna 'LOCAL' padronizada com sucesso!


,LOCAL
0,"(Fortaleza -, CE)"
1,"(Recife -, PE)"
2,"(Salvador -, BA)"
3,"(Olinda -, PE)"
4,"(Salvador -, BA)"
5,"(Salvador -, BA)"
6,"(São José De Mipibu -, RN)"
7,"(Salvador -, BA)"
8,"(Recife -, PE)"
9,"(Jaboatão Dos Guararapes -, PE)"



Total de localidades únicas: 118

Frequência das localidades (Top 20):
LOCAL
(Recife -, PE)                     197
(Fortaleza -, CE)                  152
(Salvador -, BA)                   107
(Aracaju -, SE)                     42
(Jaboatão Dos Guararapes -, PE)     41
(João Pessoa -, PB)                 29
(Olinda -, PE)                      27
(Belém -, PA)                       25
(Paulista -, PE)                    24
(Caruaru -, PE)                     24
(Petrolina -, PE)                   24
(Feira De Santana -, BA)            23
(São Paulo -, SP)                   22
(São Luís -, MA)                    21
(Natal -, RN)                       19
(Teresina -, PI)                    13
(Ananindeua -, PA)                  11
(Maceió -, AL)                      11
(Campina Grande -, PB)              10
(Abreu E Lima -, PE)                 9
Name: count, dtype: int64


In [ ]:
# Como os tratamentos anteriores removeram caracteres especiais (incluindo : e /),
# a URL atual pode estar no formato 'httpswwwreclameaquicombrnagem...'
# Vamos verificar se o padrão de barras duplas ainda existe ou se foi removido pela limpeza anterior.

# Caso as barras ainda existissem no DataFrame original antes da limpeza de caracteres especiais:
# df['URL'] = df['URL'].str.replace('.com.br//', '.com.br/', regex=False)

# Se a limpeza de caracteres especiais ja foi rodada, o texto esta sem barras.
# Vou adicionar um passo para garantir que a URL esteja limpa conforme sua solicitacao,
# assumindo que voce queira corrigir a string caso ela contenha o erro informado.

def corrigir_url_especifica(url):
    if isinstance(url, str):
        # Se houver '.com.br//', substitui por '.com.br/'
        return url.replace('.com.br//', '.com.br/')
    return url

df['URL'] = df['URL'].apply(corrigir_url_especifica)

print("Correção de barras duplas na coluna URL aplicada.")
display(df[['URL']].head())

Correção de barras duplas na coluna URL aplicada.


,URL
0,https://www.reclameaqui.com.br/nagem-loja-fisi...
1,https://www.reclameaqui.com.br/nagem-loja-fisi...
2,https://www.reclameaqui.com.br/nagem-loja-fisi...
3,https://www.reclameaqui.com.br/nagem-loja-fisi...
4,https://www.reclameaqui.com.br/nagem-loja-fisi...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   ID             1000 non-null   int64 
 1   TEMA           1000 non-null   object
 2   LOCAL          1000 non-null   object
 3   TEMPO          1000 non-null   object
 4   CATEGORIA      1000 non-null   object
 5   STATUS         1000 non-null   object
 6   DESCRICAO      1000 non-null   object
 7   URL            1000 non-null   object
 8   ANO            1000 non-null   int64 
 9   MES            1000 non-null   int64 
 10  DIA            1000 non-null   int64 
 11  DIA_DO_ANO     1000 non-null   int64 
 12  SEMANA_DO_ANO  1000 non-null   int64 
 13  DIA_DA_SEMANA  1000 non-null   int64 
 14  TRIMETRES      1000 non-null   int64 
 15  CASOS          1000 non-null   int64 
 16  LABEL          999 non-null    object
dtypes: int64(9), object(8)
memory usage: 132.9+ KB


In [ ]:
# Verificar a contagem de valores nulos por coluna
print('Valores nulos antes do tratamento:')
print(df.isnull().sum())

# Remover linhas onde as colunas essenciais sejam nulas e preencher o resto
df = df.dropna(subset=['TEMA', 'DESCRICAO'])
df = df.fillna('Não informado')

print('\nValores nulos após o tratamento:')
print(df.isnull().sum())

Valores nulos antes do tratamento:
ID               0
TEMA             0
LOCAL            0
TEMPO            0
CATEGORIA        0
STATUS           0
DESCRICAO        0
URL              0
ANO              0
MES              0
DIA              0
DIA_DO_ANO       0
SEMANA_DO_ANO    0
DIA_DA_SEMANA    0
TRIMETRES        0
CASOS            0
LABEL            1
dtype: int64

Valores nulos após o tratamento:
ID               0
TEMA             0
LOCAL            0
TEMPO            0
CATEGORIA        0
STATUS           0
DESCRICAO        0
URL              0
ANO              0
MES              0
DIA              0
DIA_DO_ANO       0
SEMANA_DO_ANO    0
DIA_DA_SEMANA    0
TRIMETRES        0
CASOS            0
LABEL            0
dtype: int64


In [ ]:
def remover_acentos_e_especiais(texto):
    if not isinstance(texto, str):
        texto = str(texto)
    # Normaliza para decompor caracteres acentuados (ex: á -> a + ´)
    nfkd_form = unicodedata.normalize('NFKD', texto)
    # Filtra mantendo apenas caracteres que não são marcas de acentuação e permitindo letras/números
    apenas_texto = "".join([c for c in nfkd_form if not unicodedata.combining(c)])
    # Remove outros caracteres especiais mantendo espaços, letras e números
    return re.sub(r'[^a-zA-Z0-9\s]', '', apenas_texto)

In [ ]:
# Identificar colunas de texto (object)
text_columns_for_special_chars = df.select_dtypes(include=['object']).columns

# Aplicar a substituição correta
for col in text_columns_for_special_chars:
    df[col] = df[col].apply(remover_acentos_e_especiais)

print(f"Sucesso! Acentos convertidos e caracteres especiais removidos nas colunas: {list(text_columns_for_special_chars)}")

# Exibir o resultado para conferência (ex: gravata)
display(df[text_columns_for_special_chars].head())

Sucesso! Acentos convertidos e caracteres especiais removidos nas colunas: ['TEMA', 'LOCAL', 'TEMPO', 'CATEGORIA', 'STATUS', 'DESCRICAO', 'URL', 'LABEL']


,TEMA,LOCAL,TEMPO,CATEGORIA,STATUS,DESCRICAO,URL,LABEL
0,Seguro Nagem celular e uma Editado pelo Reclam...,Fortaleza CE,20170108,Nagem Loja FisicaProblemas com o Atendimento,Nao resolvido,Comprei um celular LG na loja NAGEM do shoppin...,httpswwwreclameaquicombrnagemlojafisicaseguron...,Fortaleza CE
1,Demora na finalizacao da venda Mau atendimento,Recife PE,20170109,Nagem Loja Fisica,Resolvido,Fui na loja Nagem do Shopping Recife fiz uma c...,httpswwwreclameaquicombrnagemlojafisicademoran...,Recife PE
2,Problemas com a troca de produto,Salvador BA,20170110,Nagem Loja Fisica,Respondida,Comprei um celular da Samsung J5 prime no saba...,httpswwwreclameaquicombrnagemlojafisicaproblem...,Salvador BA
3,Produto com defeitodesrespeito com o consumidor,Olinda PE,20170111,Nagem Loja Fisica,Respondida,No dia 25 10 2017 realizei uma compra de um ap...,httpswwwreclameaquicombrnagemlojafisicaproduto...,Olinda PE
4,Descaso Comsumidor,Salvador BA,20170112,TVBlackfridayTela trincadaEletroeletronicosNag...,Nao resolvido,Na sexta feira do dia 24112017fui efetuar uma ...,httpswwwreclameaquicombrnagemlojafisicadescaso...,Salvador BA


In [ ]:
# 1. Converter a coluna TEMPO separadamente (pois ela é a nossa referência de data)
if 'TEMPO' in df.columns:
    df['TEMPO'] = pd.to_datetime(df['TEMPO'], errors='coerce')

# 2. Identificar automaticamente todas as colunas NUMÉRICAS
# Isso pega int64 e float64 de uma vez só
num_columns = df.select_dtypes(include=['number']).columns

# 3. Aplicar a blindagem em todas as colunas numéricas encontradas
for col in num_columns:
    # Converte para numérico, limpa erros, preenche vazios com 0 e vira Inteiro
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# 4. Verificar os resultados
print(f"Colunas numéricas tratadas: {list(num_columns)}")
print("\nNovos tipos de dados:")
print(df.dtypes)
display(df.head())

Colunas numéricas tratadas: ['ID', 'ANO', 'MES', 'DIA', 'DIA_DO_ANO', 'SEMANA_DO_ANO', 'DIA_DA_SEMANA', 'TRIMETRES', 'CASOS']

Novos tipos de dados:
ID                        int64
TEMA                     object
LOCAL                    object
TEMPO            datetime64[ns]
CATEGORIA                object
STATUS                   object
DESCRICAO                object
URL                      object
ANO                       int64
MES                       int64
DIA                       int64
DIA_DO_ANO                int64
SEMANA_DO_ANO             int64
DIA_DA_SEMANA             int64
TRIMETRES                 int64
CASOS                     int64
LABEL                    object
dtype: object


,ID,TEMA,LOCAL,TEMPO,CATEGORIA,STATUS,DESCRICAO,URL,ANO,MES,DIA,DIA_DO_ANO,SEMANA_DO_ANO,DIA_DA_SEMANA,TRIMETRES,CASOS,LABEL
0,27859289,Seguro Nagem celular e uma Editado pelo Reclam...,Fortaleza CE,2017-01-08,Nagem Loja FisicaProblemas com o Atendimento,Nao resolvido,Comprei um celular LG na loja NAGEM do shoppin...,httpswwwreclameaquicombrnagemlojafisicaseguron...,2017,1,8,8,1,6,1,1,Fortaleza CE
1,28562465,Demora na finalizacao da venda Mau atendimento,Recife PE,2017-01-09,Nagem Loja Fisica,Resolvido,Fui na loja Nagem do Shopping Recife fiz uma c...,httpswwwreclameaquicombrnagemlojafisicademoran...,2017,1,9,9,2,0,1,1,Recife PE
2,29249337,Problemas com a troca de produto,Salvador BA,2017-01-10,Nagem Loja Fisica,Respondida,Comprei um celular da Samsung J5 prime no saba...,httpswwwreclameaquicombrnagemlojafisicaproblem...,2017,1,10,10,2,1,1,1,Salvador BA
3,29999183,Produto com defeitodesrespeito com o consumidor,Olinda PE,2017-01-11,Nagem Loja Fisica,Respondida,No dia 25 10 2017 realizei uma compra de um ap...,httpswwwreclameaquicombrnagemlojafisicaproduto...,2017,1,11,11,2,2,1,1,Olinda PE
4,30812099,Descaso Comsumidor,Salvador BA,2017-01-12,TVBlackfridayTela trincadaEletroeletronicosNag...,Nao resolvido,Na sexta feira do dia 24112017fui efetuar uma ...,httpswwwreclameaquicombrnagemlojafisicadescaso...,2017,1,12,12,2,3,1,1,Salvador BA


In [ ]:
df.describe()

,ID,TEMPO,ANO,MES,DIA,DIA_DO_ANO,SEMANA_DO_ANO,DIA_DA_SEMANA,TRIMETRES,CASOS
count,1.000000e+03,1000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,8.428358e+07,2019-11-08 15:20:09.600000,2019.345000,6.639000,15.571000,186.445000,27.069000,2.700000,2.538000,1.608000
min,2.433489e+07,2017-01-08 00:00:00,2017.000000,1.000000,1.000000,2.000000,1.000000,0.000000,1.000000,1.000000
25%,3.589178e+07,2018-06-13 06:00:00,2018.000000,4.000000,8.000000,99.500000,15.000000,1.000000,2.000000,1.000000
50%,9.733731e+07,2019-11-13 00:00:00,2019.000000,7.000000,16.000000,185.500000,27.000000,3.000000,3.000000,1.000000
75%,1.195213e+08,2021-02-12 18:00:00,2021.000000,10.000000,23.000000,275.000000,40.000000,4.000000,4.000000,2.000000
max,1.501012e+08,2022-12-08 00:00:00,2022.000000,12.000000,31.000000,364.000000,53.000000,6.000000,4.000000,4.000000
std,4.296011e+07,NaN,1.626229,3.384396,8.823197,103.732519,14.837263,1.929176,1.098983,0.762179


In [ ]:
# 1. Adicionar os estados originais como uma coluna temporária para usar como rótulos
estados_originais = df['LOCAL']

# Handle the specific case: '--- - MA' or '-- - MA' should become 'Não Informado - MA'
# This regex looks for one or more hyphens/spaces at the beginning of the city part,
# followed by ' - ', then two uppercase letters for the state.
# It replaces the hyphen/space part with 'Não Informado', preserving the state.
df['LOCAL'] = df['LOCAL'].apply(
    lambda x: re.sub(r'^[-\s]+\s*-\s*([A-Z]{2})$', r'Não Informado - \1', x)
    if isinstance(x, str) else x
)

# This line was already present and handles standalone '--' if it appears as a full entry.
df['LOCAL'] = df['LOCAL'].replace('--', 'Não Informado')

# 2. Contar a frequência de cada local usando a nova coluna de display
local_counts = df['LOCAL'].value_counts().reset_index()
local_counts.columns = ['LOCAL', 'COUNT']

# 3. Calcular a porcentagem
local_counts['PERCENTAGE'] = (local_counts['COUNT'] / local_counts['COUNT'].sum()) * 100

# Use all localities directly without filtering or grouping as 'Outros'
final_local_data = local_counts

# 4. Criar o gráfico de pizza
fig = px.pie(final_local_data,
             values='PERCENTAGE',
             names='LOCAL',
             title='Distribuição Percentual de Reclamações por Localidade',
             hover_data=['COUNT'],
             labels={'LOCAL': 'Localidade', 'PERCENTAGE': 'Percentual', 'COUNT': 'Contagem'})

fig.update_traces(textposition='inside', textinfo='percent+label')

fig.show()

# 5. Remover a coluna temporária após o uso
df.drop(columns=['LOCAL'], inplace=True)

In [ ]:
# Para simplificar a visualização da coluna 'DESCRICAO', vamos truncá-la para exibição.
# Esta operação cria uma coluna temporária para fins de display e não modifica o DataFrame original.

df_truncated_description = df.copy()
df_truncated_description['DESCRICAO_TRUNCADA'] = df_truncated_description['DESCRICAO'].apply(lambda x: x[:150] + '...' if len(x) > 150 else x)

# Exibindo o cabeçalho do DataFrame com a descrição truncada
df_truncated_description.head()

,ID,TEMA,TEMPO,CATEGORIA,STATUS,DESCRICAO,URL,ANO,MES,DIA,DIA_DO_ANO,SEMANA_DO_ANO,DIA_DA_SEMANA,TRIMETRES,CASOS,LABEL,DESCRICAO_TRUNCADA
0,27859289,Seguro Nagem celular e uma Editado pelo Reclam...,2017-01-08,Nagem Loja FisicaProblemas com o Atendimento,Nao resolvido,Comprei um celular LG na loja NAGEM do shoppin...,httpswwwreclameaquicombrnagemlojafisicaseguron...,2017,1,8,8,1,6,1,1,Fortaleza CE,Comprei um celular LG na loja NAGEM do shoppin...
1,28562465,Demora na finalizacao da venda Mau atendimento,2017-01-09,Nagem Loja Fisica,Resolvido,Fui na loja Nagem do Shopping Recife fiz uma c...,httpswwwreclameaquicombrnagemlojafisicademoran...,2017,1,9,9,2,0,1,1,Recife PE,Fui na loja Nagem do Shopping Recife fiz uma c...
2,29249337,Problemas com a troca de produto,2017-01-10,Nagem Loja Fisica,Respondida,Comprei um celular da Samsung J5 prime no saba...,httpswwwreclameaquicombrnagemlojafisicaproblem...,2017,1,10,10,2,1,1,1,Salvador BA,Comprei um celular da Samsung J5 prime no saba...
3,29999183,Produto com defeitodesrespeito com o consumidor,2017-01-11,Nagem Loja Fisica,Respondida,No dia 25 10 2017 realizei uma compra de um ap...,httpswwwreclameaquicombrnagemlojafisicaproduto...,2017,1,11,11,2,2,1,1,Olinda PE,No dia 25 10 2017 realizei uma compra de um ap...
4,30812099,Descaso Comsumidor,2017-01-12,TVBlackfridayTela trincadaEletroeletronicosNag...,Nao resolvido,Na sexta feira do dia 24112017fui efetuar uma ...,httpswwwreclameaquicombrnagemlojafisicadescaso...,2017,1,12,12,2,3,1,1,Salvador BA,Na sexta feira do dia 24112017fui efetuar uma ...


Para um cruzamento entre `STATUS` e `LOCAL`, podemos gerar um gráfico semelhante:

In [ ]:
df['STATUS'] = df['STATUS'].replace('Em replica', 'Em réplica')
df['LOCAL_ORIGINAL'] = estados_originais

# 2. Agrupar por STATUS e pelo LOCAL_ORIGINAL
status_local_counts = df.groupby(['STATUS', 'LOCAL_ORIGINAL']).size().reset_index(name='COUNT')

# 3. Filtrar para os top 10 locais originais para melhor visualização
top_n_locals = df['LOCAL_ORIGINAL'].value_counts().nlargest(10).index
status_local_counts_filtered = status_local_counts[status_local_counts['LOCAL_ORIGINAL'].isin(top_n_locals)]

# 4. Gerar o gráfico
fig = px.bar(status_local_counts_filtered,
             x='LOCAL_ORIGINAL',
             y='COUNT',
             color='STATUS',
             title='Contagem de Status por Local (Dados Originais - Top 10)',
             labels={'LOCAL_ORIGINAL': 'Local Original', 'COUNT': 'Contagem', 'STATUS': 'Status'},
             barmode='group')

fig.update_layout(xaxis_tickangle=-45)
fig.show()

# Opcional: Remover a coluna temporária após o uso
df.drop(columns=['LOCAL_ORIGINAL'], inplace=True)

## Segunda parte: Análise Exploratória de Dados (EDA)

### Análise Descritiva e Sazonalidade

In [ ]:
# Distribuição de reclamações por ano
reclamacoes_por_ano = df.groupby('ANO')['CASOS'].sum().reset_index()
reclamacoes_por_ano.columns = ['ANO', 'TOTAL_RECLAMACOES']

fig_ano = px.bar(reclamacoes_por_ano, x='ANO', y='TOTAL_RECLAMACOES',
                 title='Total de Reclamações por Ano',
                 labels={'ANO': 'Ano', 'TOTAL_RECLAMACOES': 'Total de Reclamações'},
                 text='TOTAL_RECLAMACOES')
fig_ano.update_traces(textposition='outside')
fig_ano.show()

In [ ]:
# Sazonalidade: Reclamações por mês (agregado de todos os anos)
meses_nomes = {1:'Jan', 2:'Fev', 3:'Mar', 4:'Abr', 5:'Mai', 6:'Jun',
               7:'Jul', 8:'Ago', 9:'Set', 10:'Out', 11:'Nov', 12:'Dez'}

reclamacoes_por_mes = df.groupby('MES')['CASOS'].sum().reset_index()
reclamacoes_por_mes.columns = ['MES', 'TOTAL']
reclamacoes_por_mes['MES_NOME'] = reclamacoes_por_mes['MES'].map(meses_nomes)

fig_mes = px.bar(reclamacoes_por_mes, x='MES_NOME', y='TOTAL',
                 title='Sazonalidade: Reclamações por Mês (Todos os Anos)',
                 labels={'MES_NOME': 'Mês', 'TOTAL': 'Total de Reclamações'},
                 text='TOTAL')
fig_mes.update_traces(textposition='outside')
fig_mes.update_layout(xaxis={'categoryorder': 'array', 'categoryarray': list(meses_nomes.values())})
fig_mes.show()

In [ ]:
# Sazonalidade: Reclamações por dia da semana
dias_nomes = {0:'Segunda', 1:'Terça', 2:'Quarta', 3:'Quinta',
              4:'Sexta', 5:'Sábado', 6:'Domingo'}

reclamacoes_por_dia = df.groupby('DIA_DA_SEMANA')['CASOS'].sum().reset_index()
reclamacoes_por_dia.columns = ['DIA_DA_SEMANA', 'TOTAL']
reclamacoes_por_dia['DIA_NOME'] = reclamacoes_por_dia['DIA_DA_SEMANA'].map(dias_nomes)

fig_dia = px.bar(reclamacoes_por_dia, x='DIA_NOME', y='TOTAL',
                 title='Sazonalidade: Reclamações por Dia da Semana',
                 labels={'DIA_NOME': 'Dia da Semana', 'TOTAL': 'Total de Reclamações'},
                 text='TOTAL', color='TOTAL',
                 color_continuous_scale='Reds')
fig_dia.update_traces(textposition='outside')
fig_dia.update_layout(xaxis={'categoryorder': 'array',
                             'categoryarray': list(dias_nomes.values())})
fig_dia.show()

In [ ]:
# Série Temporal: Evolução mensal de reclamações com Média Móvel
df['TEMPO'] = pd.to_datetime(df['TEMPO'], errors='coerce')
df['ANO_MES'] = df['TEMPO'].dt.to_period('M')

serie_temporal = df.groupby('ANO_MES')['CASOS'].sum().reset_index()
serie_temporal['ANO_MES'] = serie_temporal['ANO_MES'].astype(str)
serie_temporal['MEDIA_MOVEL_3'] = serie_temporal['CASOS'].rolling(window=3, min_periods=1).mean()

fig_serie = go.Figure()
fig_serie.add_trace(go.Scatter(x=serie_temporal['ANO_MES'], y=serie_temporal['CASOS'],
                               mode='lines+markers', name='Reclamações',
                               line=dict(color='royalblue')))
fig_serie.add_trace(go.Scatter(x=serie_temporal['ANO_MES'], y=serie_temporal['MEDIA_MOVEL_3'],
                               mode='lines', name='Média Móvel (3 meses)',
                               line=dict(color='red', dash='dash')))
fig_serie.update_layout(title='Evolução Mensal de Reclamações com Média Móvel',
                        xaxis_title='Ano-Mês', yaxis_title='Nº de Reclamações',
                        xaxis_tickangle=-45)
fig_serie.show()

### Cruzamento de Dados: STATUS vs CATEGORIA

In [ ]:
# Cruzamento STATUS vs CATEGORIA
status_cat = df.groupby(['STATUS', 'CATEGORIA']).size().reset_index(name='COUNT')

# Top 10 categorias para melhor visualização
top_categorias = df['CATEGORIA'].value_counts().nlargest(10).index
status_cat_filtered = status_cat[status_cat['CATEGORIA'].isin(top_categorias)]

fig_status_cat = px.bar(status_cat_filtered, x='CATEGORIA', y='COUNT',
                        color='STATUS', barmode='group',
                        title='Cruzamento: Status vs Categoria (Top 10 Categorias)',
                        labels={'CATEGORIA': 'Categoria', 'COUNT': 'Contagem', 'STATUS': 'Status'})
fig_status_cat.update_layout(xaxis_tickangle=-45)
fig_status_cat.show()

In [ ]:
# 1. Filtrar categorias com volume mínimo (pelo menos 5 reclamações)
category_counts = df['CATEGORIA'].value_counts()
categories_min = category_counts[category_counts >= 5].index
df_f = df[df['CATEGORIA'].isin(categories_min)].copy()

# 2. Calcular frequências de STATUS
res = df_f.groupby('CATEGORIA')['STATUS'].value_counts(normalize=True).unstack(fill_value=0)
counts = df_f.groupby('CATEGORIA').size().rename('TOTAL_RECLAMACOES')

# 3. Identificar a coluna exata de 'Resolvido' (case-insensitive e strip)
col_ok = [c for c in res.columns if c.strip().lower() == 'resolvido']

if col_ok:
    res['TAXA_RESOLUCAO'] = res[col_ok[0]]
else:
    res['TAXA_RESOLUCAO'] = 0

# Unir com a contagem total para dar contexto ao gráfico
res_plot = res.join(counts).sort_values(['TAXA_RESOLUCAO', 'TOTAL_RECLAMACOES'], ascending=False).head(15).reset_index()

# 4. Plotar com escala de cores e rótulos claros
fig = px.bar(res_plot,
             x='TAXA_RESOLUCAO',
             y='CATEGORIA',
             orientation='h',
             title='Top 15 Categorias por Taxa de Resolução (Min. 5 Reclamações)',
             labels={'TAXA_RESOLUCAO': 'Taxa de Resolução', 'CATEGORIA': 'Categoria'},
             color='TAXA_RESOLUCAO',
             color_continuous_scale='RdYlGn',
             hover_data=['TOTAL_RECLAMACOES'])

fig.update_traces(text=res_plot.apply(lambda row: f"{row['TAXA_RESOLUCAO']:.1%} ({int(row['TOTAL_RECLAMACOES'])} qts)", axis=1),
                  textposition='outside')

fig.update_layout(xaxis_tickformat='.0%', xaxis_range=[0, 1.2])
fig.show()

In [ ]:
# Proporção de reclamações por STATUS
status_counts = df['STATUS'].value_counts().reset_index()
status_counts.columns = ['STATUS', 'COUNT']

fig_status = px.pie(status_counts, values='COUNT', names='STATUS',
                    title='Distribuição de Reclamações por Status',
                    hole=0.4)
fig_status.update_traces(textposition='inside', textinfo='percent+label')
fig_status.show()

In [ ]:
# Sazonalidade por Trimestre
trimestre_nomes = {1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}
rec_trimestre = df.groupby('TRIMETRES')['CASOS'].sum().reset_index()
rec_trimestre.columns = ['TRIMESTRE', 'TOTAL']
rec_trimestre['TRIM_NOME'] = rec_trimestre['TRIMESTRE'].map(trimestre_nomes)

fig_trim = px.bar(rec_trimestre, x='TRIM_NOME', y='TOTAL',
                  title='Reclamações por Trimestre',
                  labels={'TRIM_NOME': 'Trimestre', 'TOTAL': 'Total de Reclamações'},
                  text='TOTAL', color='TOTAL',
                  color_continuous_scale='Blues')
fig_trim.update_traces(textposition='outside')
fig_trim.show()

In [ ]:
# Análise do tamanho dos textos das reclamações
df['TAMANHO_DESCRICAO'] = df['DESCRICAO'].apply(len)

print('Estatísticas do tamanho das descrições:')
print(df['TAMANHO_DESCRICAO'].describe())

# Boxplot: Tamanho da descrição por STATUS
fig_box = px.box(df, x='STATUS', y='TAMANHO_DESCRICAO',
                 title='Distribuição do Tamanho das Reclamações por Status',
                 labels={'STATUS': 'Status', 'TAMANHO_DESCRICAO': 'Nº de Caracteres'},
                 color='STATUS')
fig_box.show()

Estatísticas do tamanho das descrições:
count     1000.000000
mean      1047.673000
std        913.740552
min          1.000000
25%        475.750000
50%        811.500000
75%       1312.500000
max      10826.000000
Name: TAMANHO_DESCRICAO, dtype: float64


In [ ]:
# Histograma da distribuição do tamanho dos textos
fig_hist = px.histogram(df, x='TAMANHO_DESCRICAO', nbins=30,
                        title='Distribuição do Tamanho dos Textos das Reclamações',
                        labels={'TAMANHO_DESCRICAO': 'Nº de Caracteres', 'count': 'Frequência'},
                        color_discrete_sequence=['steelblue'])
fig_hist.update_layout(yaxis_title='Frequência')
fig_hist.show()

### Rascunho das Métricas para o Dashboard

**Filtros Globais Planejados:**
- Estado (UF)
- Status da reclamação
- Faixa de tamanho do texto da reclamação

**Componentes Visuais e Estatísticos Planejados:**

1. **Série Temporal com Tendência** — Evolução mensal de reclamações + linha de Média Móvel
2. **Mapa Coroplético do Brasil** — Quantidade de reclamações por estado, com seletor de ano
3. **Gráfico de Pareto por Estado** — Distribuição de frequência de reclamações por UF
4. **Proporção de Resoluções** — Frequência de reclamações por tipo de STATUS (gráfico de pizza/donut)
5. **Boxplot/Histograma de Tamanho de Texto** — Distribuição do tamanho de DESCRICAO cruzada com STATUS
6. **WordCloud (NLP)** — Palavras mais frequentes nas descrições, com remoção de stopwords via NLTK

**KPIs Planejados:**
- Total de reclamações
- Taxa de resolução geral (%)
- Estado com mais reclamações
- Categoria mais reclamada

### Recomendações de Ações Estratégicas
Essas recomendações mantêm o foco nos 4 objetivos definidos neste notebook: gargalos regionais, eficiência de atendimento, sazonalidade e categorização de falhas; já que os tratamentos e as análises se basearam nesses pilares.

As recomendações consideram também que os dados representam a realidade, ou seja, consideramos a concentração de reclamações nos estados específicos, os motivos principais, os status e outras informações como realmente fidedignas para construir essas recomendações.

1.	Ação regional em Pernambuco, Ceará e Bahia. Esses três estados concentram o maior volume de reclamações. Na base, Pernambuco aparece muito à frente, seguido por Ceará e Bahia e isso justifica plano de ação regionalizado, começando por Recife, Fortaleza e Salvador.
2.	Avaliação recorrente dos casos mais recentes em “Não resolvido”, buscando entender os gargalos e direcionar ações para as tratativas dos problemas, focando em eliminar os problemas novos para que não se estendam a outras ocorrências.
3.	Avaliação de esforços para os motivos históricos principais que se mantém ao longo do tempo, que demonstram problemas recorrentes.
Os temas mais repetidos giram em torno de garantia estendida, troca de produto, produto com defeito, nota fiscal e propaganda enganosa. Isso deve direcionar a atuação com ações de discurso comercial na venda, política de troca, comunicação sobre garantia, documentação fiscal.
4.	Implementação cultural de monitoramento das análises elaboradas, que devem ser visitadas em alguma rotina gerencial. O ideal é monitorar onde aumentou o volume de reclamações, onde piorou a resolução e associar isso com quais ações foram tomadas.

